# SOLiD-PINN v0.8.8 — Missing physics recovery
CPU, Run all. No GitHub token, no private data. Tests first; 108 evaluations and 180 candidates, including 18 nominal fixed evaluations. Compare incomplete/calibrated physics, non-neural closures, neural RBF closure and oracle. Negative findings and null control retained. Save to Drive and download JSON. This is a synthetic solver-integrated closure benchmark, not a collocation PINN.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import subprocess,sys,os,tempfile,json,hashlib,math
drive.mount("/content/drive")
REV="43847936d96edc5f16f9692557face317155ecb9"
REPO=Path(tempfile.mkdtemp(prefix="solid-pinn-v088-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(REPO)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=REPO,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()==REV
hashes={"missing_physics.py":"4dedec06a174a7f1df04b5b147e181cb8fc743a56f88d915f7d249e629768426","test_missing_physics.py":"7ac81a27c8ce28c92da7439b185090e4257b976da8e31d554f7c85de8b60d174","dynamic_benchmark.py":"526d3ab253e77e1adf8cbc3b1dc1a56b8317e5e0d9b4210814fa9479a799fc73"}
for name,digest in hashes.items(): assert hashlib.sha256((REPO/"scripts"/name).read_bytes()).hexdigest()==digest
env=os.environ.copy()
env.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","scipy==1.17.0","pytest"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_missing_physics.py"],cwd=REPO,env=env,check=True)
out=REPO/"results.json"
subprocess.run([sys.executable,"scripts/missing_physics.py","--out",str(out)],cwd=REPO,env=env,check=True)
d=json.loads(out.read_text())
expected={(t,n,s,m) for t in ("null","rest","peak") for n in (1,5) for s in (11,23,37) for m in ("nominal","calibrated","constant","linear","hybrid_rbf","oracle")}
assert len(d["metrics"])==108 and {(x["truth"],x["noise"],x["seed"],x["method"]) for x in d["metrics"]}==expected
expected_candidates={(t,n,s,m,l) for t,n,s,m in expected for l in ((0,10,50) if m in ("linear","hybrid_rbf") else (0,))}
assert len(d["candidates"])==180 and {(x["truth"],x["noise"],x["seed"],x["method"],x["lam"]) for x in d["candidates"]}==expected_candidates
assert len(d["checks"])==3 and all(x["state_rmse"]<1e-6 for x in d["checks"])
for row in d["metrics"]:
    assert 0<=row["min_state"]<=row["max_state"]<=1
    group=[x for x in d["candidates"] if all(x[k]==row[k] for k in ("truth","noise","seed","method")) and x["success"]]
    assert group and row["lam"]==min(group,key=lambda x:x["val_rmse_mv"])["lam"]
d["source_revision"]=REV
d["source_hashes"]=hashes
d["targeted_tests_passed"]=5
raw=(json.dumps(d,sort_keys=True,indent=2,allow_nan=False)+"\n").encode()
sha=hashlib.sha256(raw).hexdigest()
root=Path("/content/drive/MyDrive/SOLiD-PINN/colab_runs/v088/handoff")
root.mkdir(parents=True,exist_ok=True)
dest=root/f"SOLiD-PINN_v088_colab_{sha[:20]}.json"
if dest.exists(): assert dest.read_bytes()==raw
else:
    part=dest.with_suffix(".part");part.write_bytes(raw);part.replace(dest)
assert dest.read_bytes()==raw
print("DRIVE VERIFIED",dest.name,sha)
files.download(str(dest))